**Tabela** | gold_ecommerce_itens_pedido |

**Origem** | squad2/silver/ecommerce_itens_pedido (Delta) |

 **Destino** | squad2/gold/gold_ecommerce_itens_pedido (Delta + SQL Server) |

 **Regras de Negócio Aplicadas** | 
 
 * Regra 6: SKUs únicos vendidos na última hora
 * Regra 7: Top 5 SKUs mais vendidos em 30 minutos
 * Regra 8: Receita Bruta vs Líquida e Alerta de Desconto > 25%
 * Regra 9: Alerta de desconto_aplicado > preco_unitario
 * Regra 10: Número médio de itens por pedido (Alerta < 3)

In [0]:
%pip install sqlalchemy pyodbc

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime, timedelta
import json
from sqlalchemy import create_engine

TABELA = "ecommerce_itens_pedido"

#  Caminhos oficiais no Storage
path_silver    = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"
path_gold_kpi  = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}_kpi"
path_gold_hist = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}_historico"
path_control   = f"gold/control/{TABELA}.json"

print(f" Lendo de (Silver): {path_silver}")
print(f" Gravando KPI (Overwrite): {path_gold_kpi}")
print(f" Gravando Histórico (Append): {path_gold_hist}")

# Configuração de conexão do SQL Server do projeto
SQL_SERVER = "seu-servidor.database.windows.net"
SQL_DB     = "seu-banco"
SQL_USER   = "seu-usuario"
SQL_PASS   = "sua-senha"
conn_string = f"mssql+pyodbc://{SQL_USER}:{SQL_PASS}@{SQL_SERVER}/{SQL_DB}?driver=ODBC+Driver+18+for+SQL+Server"

try:
    #  BLINDAGEM DEFENSIVA: Verifica se a tabela Silver já foi inicializada fisicamente
    if not DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
        print("\n [Aviso Gold] A tabela Silver de Itens de Pedido ainda não existe ou está vazia!")
        print(" Justificativa: Como a Silver de Pedidos ainda não rodou, 100% dos dados estão guardados de forma segura na Sala de Espera (Waiting Room).")
        print(" O processamento da Gold foi encerrado de forma limpa. Nenhuma ação é necessária por enquanto.")
    else:
        # 1. Abre a tabela Silver e converte para Pandas
        dt_silver = DeltaTable(path_silver, storage_options=get_storage_options())
        df_pandas = dt_silver.to_pandas()
        
        # Garante a tipagem de tempo correta para as janelas analíticas
        df_pandas['silver_processed_at'] = pd.to_datetime(df_pandas['silver_processed_at'])
        
        # 2. CONTROLE INCREMENTAL: Instancia o cliente do arquivo de controle
        squad2_client = get_squad2_client()
        file_client = squad2_client.get_file_client(path_control)
        
        processados = set()
        
        # Se o arquivo JSON já existir, baixa e lê o conteúdo
        if file_client.exists():
            conteudo = file_client.download_file().readall().decode('utf-8')
            processados = set(json.loads(conteudo))
        
        # Filtra apenas os dados de arquivos que a Gold ainda não processou
        df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
        
        if df_novos_dados.empty:
            print(" Camada Gold em dia! Nenhum dado novo para processar.")
        else:
            print(f" Processando {len(df_novos_dados)} novas linhas da Silver...")
            
            # 3. APLICAÇÃO DAS REGRAS DE NEGÓCIO DA PLANILHA (KPIs E ALERTAS)
            agora = df_novos_dados['silver_processed_at'].max()
            uma_hora_atras = agora - timedelta(hours=1)
            meia_hora_atras = agora - timedelta(minutes=30)
            
            # Filtros de tempo para as regras de janela
            df_1h = df_novos_dados[df_novos_dados['silver_processed_at'] >= uma_hora_atras].copy()
            df_30min = df_novos_dados[df_novos_dados['silver_processed_at'] >= meia_hora_atras].copy()
            
            # Regra 6: SKUs únicos vendidos na última hora
            skus_unicos_1h = df_1h['sku'].nunique()
            
            # Regra 7: Top 5 SKUs mais vendidos na janela de 30 minutos
            top_5_skus = df_30min.groupby('sku')['quantidade'].sum().nlargest(5).index.tolist()
            top_5_skus_str = ", ".join(top_5_skus)
            
            # Regra 8: Receita bruta vs receita líquida e Alerta de Desconto > 25%
            df_novos_dados['receita_bruta'] = df_novos_dados['quantidade'] * df_novos_dados['preco_unitario']
            df_novos_dados['receita_liquida'] = df_novos_dados['receita_bruta'] - df_novos_dados['desconto_aplicado']
            
            rec_bruta = df_novos_dados['receita_bruta'].sum()
            rec_liquida = df_novos_dados['receita_liquida'].sum()
            alerta_desconto_alto = "SIM" if (rec_bruta - rec_liquida) > (rec_bruta * 0.25) else "NAO"
            
            # Regra 9: Alerta quando desconto_aplicado > preco_unitario
            df_desc_imp = df_novos_dados[df_novos_dados['desconto_aplicado'] > df_novos_dados['preco_unitario']]
            alerta_desc_impossivel = "SIM" if not df_desc_imp.empty else "NAO"
            
            # Regra 10: Número médio de itens por pedido na última hora (Alerta se < 3)
            if not df_1h.empty:
                media_itens_1h = df_1h['quantidade'].sum() / df_1h['id_pedido'].nunique()
            else:
                media_itens_1h = 0
            alerta_queda_checkout = "SIM" if media_itens_1h < 3 and media_itens_1h > 0 else "NAO"
            
            # Estruturação do DataFrame com os resultados analíticos do lote
            df_filtrado = pd.DataFrame([{
                "data_referencia": agora.date(),
                "horario_analise": agora.time().strftime("%H:%M:%S"),
                "skus_unicos_ultima_hora": skus_unicos_1h,
                "top_5_skus_30min": top_5_skus_str,
                "receita_bruta_total": rec_bruta,
                "receita_liquida_total": rec_liquida,
                "media_itens_por_pedido_1h": round(media_itens_1h, 2),
                "alerta_desconto_excessivo_25pct": alerta_desconto_alto,
                "alerta_desconto_impossivel": alerta_desc_impossivel,
                "alerta_queda_checkout_menor_3": alerta_queda_checkout
            }])
            
            # 4. COLUNA DE AUDITORIA
            df_filtrado['gold_processed_at'] = datetime.now()
            
            # Remove os fuso-horários para o formato Delta
            for col in df_filtrado.columns:
                if pd.api.types.is_datetime64_any_dtype(df_filtrado[col]):
                    df_filtrado[col] = df_filtrado[col].dt.tz_localize(None)
                    
            # -------------------------------------------------------------------------
            # 5. ESCRITURA DOS DESTINOS (Duplo Sink)
            # -------------------------------------------------------------------------
            # A) SINK 1: TABELA DE KPIS (MODO OVERWRITE)
            write_deltalake(path_gold_kpi, df_filtrado, mode="overwrite", storage_options=get_storage_options())
            engine = create_engine(conn_string)
            df_filtrado.to_sql(name=f"{TABELA}_kpi", con=engine, if_exists='replace', index=False)
            
            # B) SINK 2: TABELA DE HISTÓRICO (MODO APPEND)
            write_deltalake(path_gold_hist, df_filtrado, mode="append", storage_options=get_storage_options())
            df_filtrado.to_sql(name=f"{TABELA}_historico", con=engine, if_exists='append', index=False)
            
            # 6. ATUALIZA O CONTROL JSON DA CAMADA GOLD
            arquivos_atuais = set(df_novos_dados['bronze_source_file'].unique())
            todos_processados = list(processados.union(arquivos_atuais))
            file_client.upload_data(json.dumps(todos_processados), overwrite=True)
            
            print(" SUCESSO! Overwrite feito em KPIs e Append feito em tabelas de Histórico do Azure e SQL Server!")

except Exception as e:
    print(f" Erro no processamento: {str(e)}")
    raise